# Lezione 45 — Uno schema per il feedback

> **Modulo:** Feedback e preference training · **Tempo stimato:** 25 minuti
> **Prerequisiti:** Lezione 22 (schema memoria), Lezione 37 (valutazione).
>
> Obiettivo pratico unico: definire uno **schema esplicito** per il feedback
> dell'utente e un validatore, la materia prima del preference training. Apre la
> Fase 7.

## Teoria essenziale

Per migliorare un sistema dal comportamento degli utenti servono dati di
**feedback** ben formati. Due grandi famiglie:

- **Assoluto** (pointwise): un giudizio su *una* risposta — pollice su/giu',
  voto 1–5. Semplice, ma la scala e' soggettiva e varia tra persone.
- **Relativo** (pairwise): "preferisco A a B". Piu' robusto, perche' confrontare
  e' piu' facile e coerente che dare un voto assoluto — ed e' la forma che
  alimenta RLHF e DPO (Lezioni 48–50).

Come per le memorie (Lezione 22), lo schema va reso **esplicito** e validato:
un feedback senza `memory_id` o con un tipo sconosciuto e' inutilizzabile.

In [1]:
from dataclasses import dataclass
from typing import Optional

TIPI_FEEDBACK = {"upvote", "downvote", "preference"}

@dataclass
class Feedback:
    feedback_id: str
    tipo: str                       # upvote | downvote | preference
    memory_id: str                  # la memoria giudicata (o la "scelta")
    rispetto_a: Optional[str] = None  # per 'preference': la memoria "rifiutata"
    peso: float = 1.0               # quanto ci fidiamo di questo feedback

def valida_feedback(fb: Feedback):
    errori = []
    if not fb.feedback_id:
        errori.append("feedback_id mancante")
    if fb.tipo not in TIPI_FEEDBACK:
        errori.append(f"tipo sconosciuto: {fb.tipo!r}")
    if not fb.memory_id:
        errori.append("memory_id mancante")
    if fb.tipo == "preference" and not fb.rispetto_a:
        errori.append("un feedback 'preference' richiede 'rispetto_a'")
    if not (0.0 <= fb.peso <= 1.0):
        errori.append(f"peso fuori range [0,1]: {fb.peso}")
    return errori

buono = Feedback("fb1", "preference", "mem_A", rispetto_a="mem_B")
print("feedback valido, errori:", valida_feedback(buono))

feedback valido, errori: []


In [2]:
# casi rotti: leggi come il validatore separa i problemi
rotti = [
    Feedback("", "upvote", "mem_A"),                      # id mancante
    Feedback("fb2", "cinque_stelle", "mem_A"),            # tipo sconosciuto
    Feedback("fb3", "preference", "mem_A"),               # manca rispetto_a
    Feedback("fb4", "upvote", "mem_A", peso=1.5),         # peso fuori range
]
for fb in rotti:
    print(f"{fb.feedback_id or '(vuoto)':8} -> {valida_feedback(fb)}")

(vuoto)  -> ['feedback_id mancante']
fb2      -> ["tipo sconosciuto: 'cinque_stelle'"]
fb3      -> ["un feedback 'preference' richiede 'rispetto_a'"]
fb4      -> ['peso fuori range [0,1]: 1.5']


## Il progetto: Memory AI Lab, passo 45

Aggiungo al progetto la raccolta del feedback come lista validata: solo i
feedback conformi entrano, gli altri vengono scartati con un motivo. E' il
guardiano all'ingresso della pipeline di preference training.

In [3]:
def raccogli(feedbacks):
    accettati, scartati = [], []
    for fb in feedbacks:
        err = valida_feedback(fb)
        (accettati if not err else scartati).append((fb, err))
    return accettati, scartati

acc, scr = raccogli([buono] + rotti)
# controllo di non-regressione
assert len(acc) == 1 and len(scr) == 4
assert all(err for _, err in scr)      # ogni scartato ha un motivo
print(f"accettati: {len(acc)}  scartati: {len(scr)} (ognuno con motivo)")

accettati: 1  scartati: 4 (ognuno con motivo)


## Riepilogo (max 8 punti)

1. Il preference training parte da **feedback ben formato**.
2. Feedback **assoluto** (voto) vs **relativo** (preferenza A>B).
3. Il relativo e' piu' robusto: confrontare e' piu' coerente che votare.
4. La forma pairwise alimenta RLHF e DPO (Lezioni 48–50).
5. Lo schema va reso **esplicito** e validato (come le memorie, Lezione 22).
6. Un `preference` senza la controparte rifiutata e' inutilizzabile.
7. Il peso codifica quanto ci fidiamo del segnale.
8. Il validatore scarta i feedback rotti con un motivo esplicito.

## Quiz

1. Perche' il feedback pairwise e' spesso piu' affidabile di un voto assoluto?
2. Cosa rende inutilizzabile un feedback di tipo `preference`?
3. A cosa serve il campo `peso`?

*(Risposte: 1. confrontare due opzioni e' piu' coerente tra persone che
assegnare un numero assoluto; 2. l'assenza della memoria "rifiutata"
(`rispetto_a`); 3. a pesare quanto ci si fida di quel feedback nell'ottimizzazione.)*